In [ ]:
import os
import subprocess
import warnings
warnings.filterwarnings('ignore')

data_dir = './lanl_data'
model_dir = './saved_models'
os.makedirs(data_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

auth_file = os.path.join(data_dir, 'auth.txt.gz')
auth_cols = ['time', 'src_user', 'dest_user', 'src_comp', 'dest_comp', 'auth_type', 'logon_type', 'auth_orientation', 'success']
auth_cache = os.path.join(data_dir, 'auth_sampled_cache.pkl')

redteam_file = os.path.join(data_dir, 'redteam.txt.gz')

print("Installing aria2 for multi-threaded downloading")
subprocess.run("sudo apt-get update > /dev/null 2>&1 && sudo apt-get install -y aria2 > /dev/null 2>&1", shell=True)

urls = {
    "redteam.txt.gz": "https://csr.lanl.gov/data-fence/1777486934/Mdxj_6cDydFPWpC4JTHCf9r689Q=/cyber1/redteam.txt.gz",
    "auth.txt.gz": "https://csr.lanl.gov/data-fence/1777486934/Mdxj_6cDydFPWpC4JTHCf9r689Q=/cyber1/auth.txt.gz"
}

for filename, url in urls.items():
    dest_path = os.path.join(data_dir, filename)
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 1024:
        continue
    print(f"Downloading {filename}...")
    cmd = ["aria2c", "-q", "-x", "16", "-s", "16", "-k", "1M", "--dir", data_dir, "--out", filename, url]
    subprocess.run(cmd)

In [ ]:
import pandas as pd
import numpy as np
import time
import gc
import os

data_dir = './lanl_data'
auth_file = os.path.join(data_dir, 'auth.txt.gz')
redteam_file = os.path.join(data_dir, 'redteam.txt.gz')
auth_cols = ['time', 'src_user', 'dest_user', 'src_comp', 'dest_comp', 'auth_type', 'logon_type', 'auth_orientation', 'success']

print("--- 1. Dynamically Analyzing Threat Data ---")
df_red = pd.read_csv(redteam_file, names=['time', 'src_user', 'src_comp', 'dest_comp'])

# LANL days start at 1. Second 0 to 86399 is "Day 1". 
df_red['day'] = (df_red['time'] // 86400) + 1

day_counts = df_red['day'].value_counts().sort_index()

# Find the first day with > 100 attacks
test_day = day_counts[day_counts > 100].index[0]
train_day = test_day - 1
val_day = test_day - 2

print(f"Automatic Dataset Split Discovered:")
print(f"  -> Validation Day : {val_day}")
print(f"  -> Training Day   : {train_day}")
print(f"  -> Testing Day    : {test_day} (Contains {day_counts[test_day]} recorded attacks)")

# Define our timestamp boundaries (Converting Days back to seconds)
# E.g., if val_day is 7, starts at 6 * 86400
T_START = (val_day - 1) * 86400
T_END = (test_day) * 86400

# Split redteam users to match auth logs
df_red[['src_user_name', 'src_domain']] = df_red['src_user'].str.split('@', n=1, expand=True)

# Create a list of tuples for the fast MultiIndex check
redteam_tuples = list(zip(df_red['src_user_name'], df_red['src_comp'], df_red['dest_comp']))

print(f"\n--- 2. Extracting & Labeling Auth Logs (Seconds {T_START} to {T_END}) ---")
start_time = time.time()
chunks = []
red_events_found = 0

# Lowered chunksize to 1 million to guarantee RAM safety
for chunk in pd.read_csv(auth_file, names=auth_cols, chunksize=1000000):
    mask = (chunk['time'] >= T_START) & (chunk['time'] < T_END)
    filtered = chunk[mask].copy()
    
    if not filtered.empty:
        # Split User@Domain
        src_split = filtered['src_user'].str.split('@', n=1, expand=True)
        filtered['src_user_name'] = src_split[0]
        filtered['src_domain'] = src_split[1] if src_split.shape[1] > 1 else 'Unknown'
        
        dest_split = filtered['dest_user'].str.split('@', n=1, expand=True)
        filtered['dest_user_name'] = dest_split[0]
        filtered['dest_domain'] = dest_split[1] if dest_split.shape[1] > 1 else 'Unknown'
        
        # --- RAM SAFE VECTORIZED RED TEAM MATCHING ---
        # We create a MultiIndex from the 3 columns and check if they exist in our redteam_tuples list
        idx_to_check = pd.MultiIndex.from_arrays([
            filtered['src_user_name'], 
            filtered['src_comp'], 
            filtered['dest_comp']
        ])
        # .isin() returns a boolean array, we cast it to int8 (0 or 1) to save RAM
        filtered['is_anomaly'] = idx_to_check.isin(redteam_tuples).astype(np.int8)
        
        red_events_found += filtered['is_anomaly'].sum()
        
        cols_to_keep = [
            'time', 'is_anomaly', 
            'src_user_name', 'src_domain', 'dest_user_name', 'dest_domain', 
            'src_comp', 'dest_comp', 'auth_type', 'logon_type', 'auth_orientation', 'success'
        ]
        
        filtered = filtered[cols_to_keep].fillna('Unknown')
        chunks.append(filtered)
        
    # Force Python to clear unused memory immediately
    del chunk, mask
    gc.collect()
        
    # LANL is chronological. If we hit a timestamp past our window, we stop.
    if filtered is not None and not filtered.empty and filtered['time'].iloc[-1] >= T_END:
        print("Passed target day boundary, stopping early to save time.")
        break

print("Concatenating chunks...")
df_auth = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

print(f"\nData extraction complete in {time.time() - start_time:.2f} seconds.")
print(f"Total events extracted: {len(df_auth)}")
print(f"Red Team events successfully matched & labeled: {red_events_found}")
display(df_auth.head())

In [ ]:
import numpy as np
from collections import Counter
import time
import gc

start_time = time.time()
print("Building Global Vocabulary from all 10 categorical columns...")

# The 10 columns that make up our "sentence"
cols_to_tokenize = [
    'src_user_name', 'src_domain', 'dest_user_name', 'dest_domain', 
    'src_comp', 'dest_comp', 'auth_type', 'logon_type', 
    'auth_orientation', 'success'
]

# 1. Count frequencies globally
global_counts = Counter()
for col in cols_to_tokenize:
    global_counts.update(df_auth[col].value_counts().to_dict())

print(f"Total unique strings found: {len(global_counts)}")

# 2. Apply < 40 frequency cutoff
FREQ_CUTOFF = 40
vocab = {
    '[PAD]': 0,
    '[MASK]': 1,
    '[OOV]': 2
}

current_idx = 3
for word, count in global_counts.items():
    if count >= FREQ_CUTOFF:
        vocab[word] = current_idx
        current_idx += 1

print(f"Vocabulary size after applying frequency cutoff (<{FREQ_CUTOFF}): {len(vocab)}")

# 3. Map strings to integers
print("Tokenizing the DataFrame...")

for col in cols_to_tokenize:
    mapped_series = df_auth[col].map(vocab)
    mapped_series = mapped_series.fillna(vocab['[OOV]'])
    df_auth[col] = mapped_series.astype(np.int32)

del global_counts
gc.collect()

print(f"Tokenization complete in {time.time() - start_time:.2f} seconds.")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import time
import gc

start_time = time.time()
print(f"Splitting data into Days {val_day}, {train_day}, and {test_day}...")

# CORRECTED TIME LOGIC: Day N starts at (N-1) * 86400
T_TRAIN_START = (train_day - 1) * 86400
T_TEST_START = (test_day - 1) * 86400

# Filter the dataframe into the 3 sets
df_val = df_auth[df_auth['time'] < T_TRAIN_START]
df_train = df_auth[(df_auth['time'] >= T_TRAIN_START) & (df_auth['time'] < T_TEST_START)]
df_test = df_auth[df_auth['time'] >= T_TEST_START]

print(f"Validation set size: {len(df_val)}")
print(f"Training set size:   {len(df_train)}")
print(f"Testing set size:    {len(df_test)}")

# Subsample Validation to save time
df_val = df_val.sample(n=min(100000, len(df_val)), random_state=42)

# For testing, we want all anomalies + 500k normal events (or however many exist)
df_anomalies = df_test[df_test['is_anomaly'] == 1]
df_normal = df_test[df_test['is_anomaly'] == 0]

sample_size = min(500000, len(df_normal))
df_normal = df_normal.sample(n=sample_size, random_state=42)

df_test = pd.concat([df_anomalies, df_normal]).sample(frac=1, random_state=42).reset_index(drop=True)

# Extract ground truth labels
y_true = df_test['is_anomaly'].values

cols_seq = [
    'src_user_name', 'src_domain', 'dest_user_name', 'dest_domain', 
    'src_comp', 'dest_comp', 'auth_type', 'logon_type', 
    'auth_orientation', 'success'
]

train_data = df_train[cols_seq].values
val_data = df_val[cols_seq].values
test_data = df_test[cols_seq].values

# Clear RAM
del df_auth, df_train, df_val, df_test, df_anomalies, df_normal
gc.collect()

print("\nDefining RAM-Optimized PyTorch Dataset...")

class FastCyberDataset(Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.long)
        
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = FastCyberDataset(train_data)
val_dataset = FastCyberDataset(val_data)
test_dataset = FastCyberDataset(test_data)

# BATCH_SIZE of 1024 is the safest for a single T4 GPU with this architecture
BATCH_SIZE = 1024 
# num_workers=0 fixes the PyTorch Multiprocessing AssertionError!
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Data Loaders ready in {time.time() - start_time:.2f} seconds.")
print(f"Number of training batches: {len(train_loader)}")

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm
import math

print("Defining Transformer and Training Loop...")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=10):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class CyberLogTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, dim_feedforward=256, dropout=0.1):
        super(CyberLogTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, max_len=10)
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, 
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.output_layer = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, src):
        src = self.embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        output = self.transformer_encoder(src)
        logits = self.output_layer(output)
        return logits

# Initialize Model
VOCAB_SIZE = len(vocab)
model = CyberLogTransformer(vocab_size=VOCAB_SIZE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Hyperparameters
EPOCHS = 2  
LEARNING_RATE = 1e-4
MASK_TOKEN_ID = vocab['[MASK]']

criterion = nn.CrossEntropyLoss(ignore_index=vocab['[PAD]'])
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = GradScaler()

def train_epoch(model, dataloader, criterion, optimizer, device, epoch):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    
    for batch_idx, sequences in enumerate(progress_bar):
        sequences = sequences.to(device)
        targets = sequences.clone()
        
        # INSTANT GPU MASKING
        curr_batch_size = sequences.size(0)
        mask_indices = torch.randint(0, 10, (curr_batch_size,), device=device)
        batch_indices = torch.arange(curr_batch_size, device=device)
        sequences[batch_indices, mask_indices] = MASK_TOKEN_ID
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast():
            outputs = model(sequences)
            outputs = outputs.permute(0, 2, 1)
            loss = criterion(outputs, targets)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        if batch_idx % 200 == 0:
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
            
    return total_loss / len(dataloader)

def validate_epoch(model, dataloader, criterion, device, epoch):
    model.eval()
    total_loss = 0.0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS} [Val]")
    
    with torch.no_grad():
        for sequences in progress_bar:
            sequences = sequences.to(device)
            targets = sequences.clone()
            
            curr_batch_size = sequences.size(0)
            mask_indices = torch.randint(0, 10, (curr_batch_size,), device=device)
            batch_indices = torch.arange(curr_batch_size, device=device)
            sequences[batch_indices, mask_indices] = MASK_TOKEN_ID
            
            with autocast():
                outputs = model(sequences)
                outputs = outputs.permute(0, 2, 1)
                loss = criterion(outputs, targets)
                
            total_loss += loss.item()
            
    return total_loss / len(dataloader)

print("\nStarting Training...")
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, epoch)
    val_loss = validate_epoch(model, val_loader, criterion, device, epoch)
    
    epoch_time = time.time() - epoch_start_time
    print(f"\nEnd of Epoch {epoch} | Time: {epoch_time:.2f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}\n")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), os.path.join(model_dir, 'auth_model.pth'))

print("Training Complete!")

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F

print("--- Starting Final Evaluation on Test Set ---")
model.eval()

# We evaluate on the test loader. 
# As per the paper, we only score fields 1, 3, 4, 5, 6, 7, 8 
# (Omitting 0:src_user, 2:dest_user, 9:success)
indices_to_score = [1, 3, 4, 5, 6, 7, 8]
MASK_ID = vocab['[MASK]']

anomaly_scores = []
y_true_list = []

print("Running Inference (Calculating Anomaly Scores)...")
with torch.no_grad():
    # We iterate over the test_dataset directly to access the labels alongside the sequences
    # We use a batch size of 1024 for manual inference
    for i in tqdm(range(0, len(test_dataset), 1024), desc="Scoring Batches"):
        # The dataset returns (sequence), but we need to fetch the original ground truth from y_true
        batch_seqs = test_dataset.data[i:i+1024].to(device)
        batch_labels = y_true[i:i+1024]
        
        batch_size = batch_seqs.size(0)
        batch_scores = torch.zeros(batch_size, device=device)
        
        # Mask each target field one by one, predict, and get the CrossEntropy Loss
        for idx in indices_to_score:
            masked_seqs = batch_seqs.clone()
            original_tokens = batch_seqs[:, idx].clone()
            
            # Mask the token
            masked_seqs[:, idx] = MASK_ID
            
            with autocast():
                logits = model(masked_seqs) 
                target_logits = logits[:, idx, :] 
                loss = F.cross_entropy(target_logits, original_tokens, reduction='none')
                
            batch_scores += loss
            
        # Average the loss across the 7 scored fields
        batch_scores = batch_scores / len(indices_to_score)
        
        anomaly_scores.extend(batch_scores.cpu().numpy())
        y_true_list.extend(batch_labels)

# Convert to numpy arrays for sklearn
anomaly_scores = np.array(anomaly_scores)
y_true_eval = np.array(y_true_list)

# 1. Calculate AUC Scores
roc_auc = roc_auc_score(y_true_eval, anomaly_scores)
pr_auc = average_precision_score(y_true_eval, anomaly_scores)

print(f"\n--- METRICS ---")
print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"PR-AUC (Average Precision): {pr_auc:.4f}")

# 2. Plot ROC Curve
fpr, tpr, thresholds = roc_curve(y_true_eval, anomaly_scores)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Transformer MLM)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)

# 3. Determine Threshold and plot Confusion Matrix
# Find the threshold that limits False Positives to 5% (simulating SOC alert fatigue constraints)
target_fpr = 0.05
idx = np.argmin(np.abs(fpr - target_fpr))
chosen_threshold = thresholds[idx]

print(f"\nSetting Alert Threshold to {chosen_threshold:.4f} (Targets ~5% False Positive Rate)")

y_pred = (anomaly_scores >= chosen_threshold).astype(int)

plt.subplot(1, 2, 2)
cm = confusion_matrix(y_true_eval, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Anomaly'], 
            yticklabels=['Actual Normal', 'Actual Anomaly'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_true_eval, y_pred, target_names=['Normal (0)', 'Red Team (1)']))